<a href="https://colab.research.google.com/github/zain4cs/Encoding_Methods/blob/main/Column_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import pandas as pd
import numpy as np

________________________

**ML Libraries**

In [31]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split

_____________

In [32]:
df = pd.read_csv("/content/drive/MyDrive/Datasets/covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [33]:
df['cough'].value_counts()

,count
cough,
Mild,62
Strong,38


In [34]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


_________________________

**Normal Way to Transform Columns**

In [35]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'], test_size=0.2)

In [36]:
X_train.head()

,age,gender,fever,cough,city
29,34,Female,NaN,Strong,Mumbai
74,34,Female,104.0,Strong,Delhi
2,42,Male,101.0,Mild,Delhi
87,47,Male,101.0,Strong,Bangalore
33,26,Female,98.0,Mild,Kolkata


___________________

Apply Simple Imputer to Fill Missign Value

In [37]:
# Adding Simple Imputer to Fever Coumn

si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

X_test_fever = si.transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

_________________

Apply Ordinal-Encoder

In [38]:
X_train['cough'].value_counts()

,count
cough,
Mild,51
Strong,29


In [39]:
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

_________________

Apply One-Hot-Encoder

In [40]:
df['city'].value_counts()

,count
city,
Kolkata,32
Bangalore,30
Delhi,22
Mumbai,16


In [41]:
ohe = OneHotEncoder(drop='first', sparse_output=True)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']]).toarray()
X_test_gender_city = ohe.transform(X_test[['gender','city']]).toarray()

X_train_gender_city.shape

(80, 4)

____________

In [42]:
X_train_age = X_train.drop(columns=['gender','fever','cough','city'])
X_test_age = X_test.drop(columns=['gender','fever','cough','city'])
X_train_age.shape

(80, 1)

In [43]:
X_train_transformed = np.concatenate((X_train_fever, X_train_cough, X_train_gender_city, X_train_age), axis=1)
X_test_transformed = np.concatenate((X_test_fever, X_test_cough, X_test_gender_city, X_test_age), axis=1)

X_train_transformed.shape

(80, 7)

___________________

**Now Apply Transformer Linrary**

In [44]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 80 entries, 29 to 63
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   age     80 non-null     int64  
 1   gender  80 non-null     object 
 2   fever   71 non-null     float64
 3   cough   80 non-null     object 
 4   city    80 non-null     object 
dtypes: float64(1), int64(1), object(3)
memory usage: 3.8+ KB


In [45]:
X_train.head()

,age,gender,fever,cough,city
29,34,Female,NaN,Strong,Mumbai
74,34,Female,104.0,Strong,Delhi
2,42,Male,101.0,Mild,Delhi
87,47,Male,101.0,Strong,Bangalore
33,26,Female,98.0,Mild,Kolkata


In [46]:
from sklearn.compose import ColumnTransformer

In [53]:
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(), ['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild','Strong']]), ['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'),['gender','city'])
], remainder='passthrough')

In [55]:
transformer.fit_transform(X_train).shape

(80, 7)

In [59]:
transformer.transform(X_test).shape

(20, 7)